# 04 · Evaluation

How well the pipeline finds contradictions that were deliberately injected into *The Wonderful Wizard of Oz* (`data/test_corpus/oz_injections.json`), measured as precision and recall. The numbers come from `python scripts/evaluate.py`, which saves them in `data/results/`; this notebook only displays them.

- **Recall:** the share of injected contradictions that were flagged.
- **Precision:** the share of flags that point at an injected contradiction.
- **Tier agreement:** how often the flag's confidence matches the tier the injection was designed for.

## 0. Setup

In [ ]:
import importlib.util
import sys

print(f"Python {sys.version.split()[0]} at {sys.executable}")
if sys.version_info < (3, 11):
    raise RuntimeError("This kernel's Python is too old: click the kernel name (top right) and choose 'Continuity Editor (.venv)'.")

# Libraries this notebook uses; continuity_editor is this project's own code.
REQUIRED = ["pandas", "ollama", "rapidfuzz", "neo4j", "pydantic", "pydantic_settings", "continuity_editor"]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    %pip install --quiet -r ../requirements.txt -e ..
    print("Done. Restart the kernel (Restart in the toolbar), then run the notebook again.")
else:
    print("All libraries are installed.")

In [ ]:
import json

import pandas as pd

from continuity_editor.paths import results_dir

pd.set_option("display.max_colwidth", 140)
path = results_dir() / "evaluation.json"
if not path.exists():
    raise FileNotFoundError("Run `python scripts/evaluate.py` in the terminal first.")
result = json.loads(path.read_text())
print(" · ".join(f"{k}: {v}" for k, v in result["meta"].items()))
pd.Series({
    "injected contradictions": result["contradictions"],
    "detected": result["detected"],
    "recall": f"{result['recall']:.0%}",
    "flags raised": result["flags"],
    "flags pointing at an injection": result["true_positive_flags"],
    "precision": "n/a" if result["precision"] is None else f"{result['precision']:.0%}",
    "false alarms on control edits": result["control_false_alarms"],
    "tier agreement": "n/a" if result["tier_agreement"] is None else f"{result['tier_agreement']:.0%}",
}, name="value").to_frame()

## Recall by category

In [ ]:
pd.DataFrame(result["by_category"]).T

## Precision by confidence tier

High-confidence flags should be the most precise.

In [ ]:
pd.DataFrame(result["by_confidence"]).T

## Every injected contradiction

In [ ]:
pd.DataFrame(result["injections"])[
    ["id", "chapter", "category", "replace", "detected", "detected_in_chapter",
     "expected_confidence", "predicted_confidence", "reason"]
].set_index("id")

## Full report

`data/results/evaluation_report.md` has the same tables plus every false alarm with the checker's reason.